# Day 1 통합 정리 연습 — 결합 + 구조변환

지금까지 배운 **결합**(`concat`·`merge`)과 **구조변환**(`pivot_table`·`melt`)을 **작은 카페 데이터**로 한 번에 되짚습니다.
데이터가 몇 줄뿐이라 **모든 행이 눈에 보여서**, 각 단계가 무엇을 하는지 바로 확인할 수 있어요.

**전체 흐름은 딱 이거예요:**
> 흩어진 작은 표들 → ① **결합**해서 하나의 통합표로 → ② **구조변환**으로 요약표 만들기

각 문제는 **문제 + 힌트 + `# 💡 풀이`** 로 되어 있습니다. 위에서부터 순서대로 실행하세요.

In [ ]:
import numpy as np
import pandas as pd

## 🗂️ 데이터 소개 — 카페 주문

세 개의 작은 표가 있습니다. **"한 행이 무엇인지"(그레인)** 가 표마다 다른 것에 주목하세요.

In [ ]:
# 주문(1·2월로 나뉘어 있음) — 한 행 = 주문 1건
orders_jan = pd.DataFrame({'order_id':['O1','O2'], 'customer_id':['C1','C2'], '주문월':['1월','1월']})
orders_feb = pd.DataFrame({'order_id':['O3','O4'], 'customer_id':['C1','C3'], '주문월':['2월','2월']})

# 주문상품 — 한 행 = 주문에 담긴 상품 1개 (한 주문에 여러 줄이 생김!)
items = pd.DataFrame({
    'order_id': ['O1','O1','O2','O3','O4','O4'],
    '상품':     ['아메리카노','케이크','라떼','쿠키','아메리카노','샌드위치'],
    '카테고리': ['음료','디저트','음료','디저트','음료','푸드'],
    '금액':     [4500, 6000, 5000, 3000, 4500, 7000],
})

# 고객 — 한 행 = 고객 1명
customers = pd.DataFrame({'customer_id':['C1','C2','C3'], '지역':['서울','부산','서울'], '등급':['GOLD','SILVER','GOLD']})

display(orders_jan); display(orders_feb); display(items); display(customers)

---
# Part 1 · 결합 되짚기 — 흩어진 표를 하나로

## ✏️ 연습 1 — `concat` 으로 1·2월 주문 잇기

1월 주문(`orders_jan`)과 2월 주문(`orders_feb`)은 **구조가 같은** 표입니다. 위아래로 이어붙여 하나의 `orders`로 만드세요.
> 힌트: `pd.concat([...], ignore_index=True)` — 인덱스를 새로 매김

In [ ]:
# 💡 풀이
orders = pd.concat([orders_jan, orders_feb], ignore_index=True)
orders   # O1~O4, 4건이 한 표로

## ✏️ 연습 2 — `merge` 로 주문에 고객정보 붙이기

`orders`에 고객의 **지역·등급**을 붙이세요. 두 표의 공통 키는 `customer_id` 입니다.
> 힌트: `orders.merge(customers, on='customer_id', how='left')` — 왼쪽(주문) 전부 유지

In [ ]:
# 💡 풀이
orders_c = orders.merge(customers, on='customer_id', how='left')
orders_c   # 주문마다 지역·등급이 붙음

## ✏️ 연습 3 — `merge` 로 통합표 완성 (그레인 주의)

이제 **주문상품(`items`)** 에 방금 만든 주문·고객 정보(`orders_c`)를 붙여 **통합표 `df`** 를 만드세요.
> 힌트: `items.merge(orders_c, on='order_id', how='left')`
>
> ⚠️ `items`는 **주문상품 단위**(6줄)라, 통합표도 6줄입니다. 한 주문(O1·O4)이 2줄씩인 것에 주목! (연습 5에서 다시 나옵니다)

In [ ]:
# 💡 풀이
df = items.merge(orders_c, on='order_id', how='left')
df   # 상품 한 줄마다 주문월·지역·등급까지 붙은 통합표 (6줄)

---
# Part 2 · 구조변환 되짚기 — 통합표로 요약표 만들기

## ✏️ 연습 4 — `pivot_table` 로 지역 × 카테고리 매출표

통합표 `df`로 **지역(행) × 카테고리(열) 매출 합계**표를 만드세요.
> 힌트: `pivot_table(index='지역', columns='카테고리', values='금액', aggfunc='sum', fill_value=0)`
> — 매출은 합계이므로 `aggfunc='sum'` 꼭!

In [ ]:
# 💡 풀이
매출표 = df.pivot_table(index='지역', columns='카테고리', values='금액', aggfunc='sum', fill_value=0)
매출표   # 서울: 음료9000·디저트9000·푸드7000 / 부산: 음료5000

## ✏️ 연습 5 — `nunique` 로 지역별 '진짜 주문 수'

지역별 주문 수를 세려고 합니다. 그런데 `df`는 **주문상품 단위**(한 주문이 여러 줄)라, 그냥 세면(`count`) 부풀려집니다.
`count`와 `nunique`를 **둘 다** 구해 차이를 확인하세요.
> 힌트: `df.groupby('지역')['order_id'].agg(줄수='count', 주문수='nunique')`

In [ ]:
# 💡 풀이
지역별주문 = df.groupby('지역')['order_id'].agg(줄수='count', 주문수='nunique')
지역별주문
# 서울: 줄수 5 (O1·O3·O4의 상품 줄) vs 실제 주문수 3 → nunique 가 맞음!

## ✏️ 연습 6 — `melt` 로 요약표를 Long 으로

연습 4의 `매출표`(Wide)를 `melt`로 **Long**으로 펴보세요. (지역·카테고리·매출 3열)
> 힌트: `매출표.reset_index().melt(id_vars='지역', var_name='카테고리', value_name='매출')`

In [ ]:
# 💡 풀이
매출_long = 매출표.reset_index().melt(id_vars='지역', var_name='카테고리', value_name='매출')
매출_long   # 옆으로 퍼졌던 카테고리가 세로로 쌓임 (Wide → Long)

---
## ✅ Day 1 한눈 정리

오늘 배운 걸 한 장으로 정리하면:

**① 결합 — 흩어진 표를 하나로**

| 도구 | 언제 | 한 줄 요약 |
|---|---|---|
| `concat` | 같은 구조를 위아래로 | 1·2월 주문 이어붙이기 |
| `merge` | 공통 키로 옆에 붙이기 | 주문 ↔ 고객, 주문상품 ↔ 주문 |
| 그레인 주의 | 한 행의 단위가 다를 때 | 주문상품(6줄) vs 주문(4건) |

**② 구조변환 — 모양 바꿔 요약**

| 도구 | 방향 | 한 줄 요약 |
|---|---|---|
| `pivot_table` | Long → Wide (집계) | 지역 × 카테고리 매출표 (`aggfunc='sum'`) |
| `nunique` | 개수 셀 때 | 주문상품 단위에선 `count` 말고 `nunique` |
| `melt` | Wide → Long | 요약표를 다시 세로로 |

> 핵심 한 문장: **결합으로 하나를 만들고, 구조변환으로 요약한다.** 이게 Day1 데이터 준비의 전부입니다.